In [1]:
import pandas as pd
import polars as pl
from tqdm import tqdm
import streamlit as st
import numpy as np


In [2]:
df=pd.read_csv('C:/Users/Danya/Downloads/train_data/realty_data.csv')
dfpl=pl.read_csv('C:/Users/Danya/Downloads/train_data/realty_data.csv')
df.head()

,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source
0,"3-комнатная, 137 м²",NaN,63000000,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН
1,"Студия, 16,7 м²",NaN,3250000,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик
2,"3-комнатная, 76 м²",NaN,16004680,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость
3,"1-комнатная, 24 м²",NaN,7841776,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М
4,"3-комнатная, 126 м²",NaN,120000000,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик


In [3]:
df['city_cat']=df['city'].astype('category')
df['price']=df['price'].astype(np.float64)
df['district_cat']=df['district'].astype('category')
df.head()

,product_name,period,price,postcode,address_name,lat,lon,object_type,total_square,rooms,floor,city,settlement,district,area,description,source,city_cat,district_cat
0,"3-комнатная, 137 м²",NaN,63000000.0,127473.0,"2-й Щемиловский переулок, 5а",55.778894,37.608844,Квартира,137.0,3.0,6.0,Москва,NaN,Тверской район,NaN,Просторная квартира свободной планировки с пан...,ЦИАН,Москва,Тверской район
1,"Студия, 16,7 м²",NaN,3250000.0,108815.0,"Харлампиева, 46",55.551025,37.313054,Квартира,16.7,NaN,1.0,Москва,NaN,Филимонковское поселение,NaN,ВНИМАНИЕ! ОЧЕНЬ ПРИВЛЕКАТЕЛЬНОЕ ПРЕ...,Домклик,Москва,Филимонковское поселение
2,"3-комнатная, 76 м²",NaN,16004680.0,NaN,"ЖК Прокшино, 8 к4",55.594802,37.431264,Квартира,76.0,3.0,6.0,Москва,NaN,Сосенское поселение,NaN,"Apт.1684018. 0,01% - гибкая ипотека! Воспользу...",Яндекс.Недвижимость,Москва,Сосенское поселение
3,"1-комнатная, 24 м²",NaN,7841776.0,NaN,"ЖК Прокшино, 6 к2",55.594332,37.428099,Квартира,24.0,1.0,10.0,Москва,NaN,Сосенское поселение,NaN,Продается однокомнатная квартира № 381 в новос...,Новострой-М,Москва,Сосенское поселение
4,"3-комнатная, 126 м²",NaN,120000000.0,121352.0,"Давыдковская, 18",55.721097,37.464342,Квартира,126.0,3.0,16.0,Москва,NaN,Фили-Давыдково район,NaN,Шикарное предложение!\nПродаётся трёхкомнатная...,Домклик,Москва,Фили-Давыдково район


In [9]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
import pickle

# Исходные данные
X = df[['lat', 'lon', 'total_square', 'floor', 'rooms', 'city_cat', 'district_cat']]
y = df['price']

# Корректируем списки признаков — убираем отсутствующие в X колонки
num_features = ['lat', 'lon', 'total_square', 'floor', 'rooms']
cat_features = ['city_cat', 'district_cat']

# Предобработка
numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, num_features),
    ('cat', categorical_transformer, cat_features)
])

# Разбиение данных
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, random_state=777
)

# Создаём пайплайн
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', GradientBoostingRegressor(n_estimators=100, random_state=777))
])

# Кросс‑валидация на обучающих данных
cv_scores = cross_val_score(full_pipeline, X_train, y_train, cv=3, scoring='r2')

# Обучение модели
full_pipeline.fit(X_train, y_train)

# Сохранение модели
with open('final_gb_model.pkl', 'wb') as f:
    pickle.dump(full_pipeline, f)

# Загрузка модели
with open('final_gb_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)

# Предсказание на тестовой выборке (используем загруженную модель)
y_pred = loaded_model.predict(X_test)

# Расчёт метрик
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)

# Формирование результатов
results = {
    'CV_R2': np.mean(cv_scores),
    'Test_R2': r2,
    'RMSE': rmse,
    'MAE': mae
}
results_df = pd.DataFrame(results, index=[0])  #результат
print("Results:")
print(results_df.round(4))


Results:
    CV_R2  Test_R2          RMSE           MAE
0  0.8751   0.8747  1.284293e+07  5.359331e+06
